In [1]:
# 运行准备：按示例代码5.1～5.3组织数据和fe，并使用示例代码5.43定义FeatSet
import os
import csv
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer

data_file = "../pybook-data/ch5/imdb_labelled.txt"
df = pd.read_csv(data_file, names=["sentence", "label"], sep="\t", quoting=csv.QUOTE_NONE)
sents = df["sentence"].values
y = df["label"].values

out_dir = "output"
train_file = f"{out_dir}/imdb_labelled_train.csv"
test_file = f"{out_dir}/imdb_labelled_test.csv"
if os.path.exists(train_file) and os.path.exists(test_file):
    print(f'Files exist at "{train_file}" and "{test_file}"')
    df_train, df_test = pd.read_csv(train_file), pd.read_csv(test_file)
    sents_train, sents_test = df_train["sentence"].values, df_test["sentence"].values
    y_train, y_test = df_train["label"].values, df_test["label"].values
else:
    os.makedirs(out_dir, exist_ok=True)
    sents_train, sents_test, y_train, y_test = train_test_split(sents, y, test_size=0.2, random_state=1)
    print(f'训练样本数量：{len(sents_train)}')
    print(f'测试样本数量：{len(sents_test)}')
    df_train = pd.DataFrame({'sentence': sents_train, 'label': y_train})
    df_test = pd.DataFrame({'sentence': sents_test, 'label': y_test})
    df_train.to_csv(train_file, index=False)
    df_test.to_csv(test_file, index=False)

fe = CountVectorizer()
fe.fit(sents_train)
vob = fe.get_feature_names_out()
print(f'词表大小：{len(vob)}')
x_train = fe.transform(sents_train)
x_test = fe.transform(sents_test)
print(f'训练集特征规模：{x_train.shape}')
print(f'测试集特征规模：{x_test.shape}')
import torch
from torch.utils.data import Dataset

class FeatSet(Dataset):
    def __init__(self, feats, labels):
        self.feats = feats
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        dense = self.feats[idx].toarray().squeeze()
        return torch.tensor(dense, dtype=torch.float32), torch.tensor(self.labels[idx], dtype=torch.long)
# 使用示例代码5.44创建训练集DataLoader
from torch.utils.data import DataLoader
train_ds = FeatSet(fe.transform(sents_train), y_train)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)

# 使用示例代码5.45定义SimpleNet
import torch.nn as nn

class SimpleNet(nn.Module):
    def __init__(self, vocab_size, num_class=2):
        super().__init__()
        self.net = nn.Linear(vocab_size, num_class)

    def forward(self, x):
        return self.net(x)

vocab_size = len(vob)
from transformers import get_linear_schedule_with_warmup

model = SimpleNet(vocab_size)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
model.to(device) #模型 → GPU
criterion = nn.CrossEntropyLoss() #设定损失函数
trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=1e-4)
max_epoch = 300
total_steps = len(train_loader) * max_epoch
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)
print("可训练参数及参数量:")
print([(name, param.numel()) for name, param in model.named_parameters() if param.requires_grad])
# 使用示例代码5.44创建测试集DataLoader
test_ds = FeatSet(fe.transform(sents_test), [0]*len(sents_test))
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)
import numpy as np

Files exist at "output/imdb_labelled_train.csv" and "output/imdb_labelled_test.csv"
词表大小：2647
训练集特征规模：(800, 2647)
测试集特征规模：(200, 2647)


/Users/xinzijie/.local/share/uv/python/cpython-3.12.13-macos-aarch64-none/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


可训练参数及参数量:
[('net.weight', 5294), ('net.bias', 2)]


## 训练SimpleNet

In [2]:
import time

t0 = time.perf_counter() # 记录开始时刻
for epoch in range(max_epoch):
    model.train()
    total_loss = 0
    for batch in train_loader:
        xb, yb = batch
        xb, yb = xb.to(device), yb.to(device)
        loss = criterion(model(xb), yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    total_loss /= len(train_loader) # 注意要在内循环之外
    if epoch % 20 == 0: # 每20轮次打印一次训练损失
        print(f"epoch {epoch:02d} : Loss = {total_loss:.4f}")
if device.type == "cuda":
    torch.cuda.synchronize(device)
total_time = time.perf_counter() - t0 # 训练结束计时
print(f"Training finished in {total_time/60:.2f} min")

epoch 00 : Loss = 0.6942


epoch 20 : Loss = 0.6441


epoch 40 : Loss = 0.6042


epoch 60 : Loss = 0.5720


epoch 80 : Loss = 0.5437


...

epoch 220 : Loss = 0.4449


epoch 240 : Loss = 0.4392


epoch 260 : Loss = 0.4362


epoch 280 : Loss = 0.4321


Training finished in 0.47 min


## SimpleNet性能评测

In [3]:
from sklearn.metrics import classification_report

model.eval() #将模型设置为推理模式
all_preds = []
with torch.no_grad(): # 禁用梯度计算，提高推理效率
    for xb, _ in test_loader:
        xb = xb.to(device)
        output = model(xb)
        preds = torch.argmax(output, dim=1).cpu().numpy() # 每个样本取得分最高的类别作为预测结果
        all_preds.extend(preds)
# 性能评测
print(classification_report(y_test, all_preds, digits=3))

              precision    recall  f1-score   support

           0      0.843     0.867     0.854       105
           1      0.848     0.821     0.834        95

    accuracy                          0.845       200
   macro avg      0.845     0.844     0.844       200
weighted avg      0.845     0.845     0.845       200



## 用额外样本测试SimpleNet

In [4]:
input_texts = ['this is a good movie', 'the movie is horrible', 'a horrible movie', 'not a bad movie']
input_ds = FeatSet(fe.transform(input_texts), [0]*len(input_texts))
input_loader = DataLoader(input_ds, batch_size=16, shuffle=False)
model.eval()
all_scores = []
with torch.no_grad():
    for batch in input_loader:
        xb, yb = batch
        xb = xb.to(device)
        yb = yb.to(device)
        output = model(xb)
        scores = torch.softmax(output, dim=1).cpu().numpy()
        all_scores.append(scores)
scores_simplenet = np.vstack(all_scores)
print(scores_simplenet)

[[0.4893473  0.51065266]
 [0.5737359  0.4262641 ]
 [0.58029246 0.41970754]
 [0.6643327  0.3356673 ]]
